In [ ]:
import json
import os
from collections import defaultdict

merged_path = os.path.join(os.getcwd(), "merged_annotations_clean.json")
with open(merged_path, "r", encoding="utf-8") as fp:
    coco = json.load(fp)

# 1. 원본 이미지 메타데이터에서 drug_shape, color_class 등 관련 키 확인
print("=== images의 첫 항목에 있는 전체 키 목록 ===")
print(list(coco["images"][0].keys()))

# 2. shape, color 관련 키워드가 들어간 키 찾기
sample_img = coco["images"][0]
shape_color_keys = [
    k
    for k in sample_img.keys()
    if "shape" in k.lower() or "color" in k.lower() or "class" in k.lower()
]
print(f"\nshape/color/class 관련 키: {shape_color_keys}")

for k in shape_color_keys:
    print(f"  {k}: {sample_img[k]}")

# 3. category_id별로 대표 이미지의 shape 정보 모아서 비교
category_id_to_name = {cat["id"]: cat["name"] for cat in coco["categories"]}
img_id_to_info = {img["id"]: img for img in coco["images"]}

cat_to_shapes = defaultdict(set)
cat_to_colors = defaultdict(set)

for ann in coco["annotations"]:
    img_info = img_id_to_info[ann["image_id"]]
    cat_id = ann["category_id"]

    for k in shape_color_keys:
        val = img_info.get(k)
        if val:
            if "shape" in k.lower():
                cat_to_shapes[cat_id].add(val)
            else:
                cat_to_colors[cat_id].add(val)

print("\n=== 클래스별 shape/color 정보 (일부) ===")
for cat_id in list(category_id_to_name.keys())[:10]:
    name = category_id_to_name[cat_id]
    shapes = cat_to_shapes.get(cat_id, set())
    colors = cat_to_colors.get(cat_id, set())
    print(f"{name} (id={cat_id}): shapes={shapes}")
